# DeepAgentには結構余計なツールがたくさん付いているので、必要なもの以外を取り除く

In [11]:
import pathlib
import uuid

from pydantic import BaseModel, Field
from deepagents import create_deep_agent
from deepagents.backends.filesystem import FilesystemBackend
from google import genai
from google.genai import types
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver

In [12]:
WORKSPACE_DIR = pathlib.Path(".") / "workspace"
SKILLS_DIR = WORKSPACE_DIR / "skills"

In [13]:
genai_client = genai.Client().aio

In [14]:
class WebSearchInput(BaseModel):
    query: str = Field(description="検索クエリ")

@tool("web_search", description="Google検索を行うツール。自然言語の質問で聞ける。", args_schema=WebSearchInput)
async def web_search(query: str):
    grounding_tool = types.Tool(
        google_search=types.GoogleSearch()
    )

    config = types.GenerateContentConfig(
        tools=[grounding_tool],
        thinking_config=types.ThinkingConfig(thinking_budget=0),  # thinking OFF
        max_output_tokens=256,
    )

    response = await genai_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=query,
        config=config,
    )

    return response.text


In [ ]:
model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")

instruction = """あなたは自宅に置かれている音声で応答する日本語のAIホームエージェントです。以下のように振る舞ってください。
- 音声エージェントであるため、ユーザーへの応答はすべて日本語で、マークダウンのように構造化された形式ではなく、自然な会話形式で行ってください。
- 音声応答は3文程度に収めてください。
"""

from langchain.agents import create_agent
from deepagents.backends import StateBackend
from deepagents.middleware.filesystem import FilesystemMiddleware
from deepagents.middleware.summarization import _compute_summarization_defaults, _DeepAgentsSummarizationMiddleware

backend = StateBackend
fs_middleware = FilesystemMiddleware(
    backend=backend,
    system_prompt="",
#     system_prompt="""## Filesystem Tools `ls`, `read_file`

# You have access to a filesystem which you can interact with using these tools.
# All file paths must start with a /.

# - ls: list files in a directory (requires absolute path)
# - read_file: read a file from the filesystem""",
)
fs_middleware.tools = [
    fs_middleware._create_read_file_tool(),
    fs_middleware._create_ls_tool(),
]

summarization_defaults = _compute_summarization_defaults(model)

middleware = [
    fs_middleware,
    _DeepAgentsSummarizationMiddleware(
        model=model,
        backend=backend,
        trigger=summarization_defaults["trigger"],
        keep=summarization_defaults["keep"],
        trim_tokens_to_summarize=None,
        truncate_args_settings=summarization_defaults["truncate_args_settings"],
    ),
]

agent = create_agent(
    model=model,
    system_prompt=instruction,
    middleware=middleware,
    tools=[web_search],
    checkpointer=MemorySaver(),
)


In [16]:
session_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

In [17]:
question = "金沢に旅行に行きたい。さいたま市からの行き方を教えて。"
messages = {"messages": [{"role": "user", "content": question}]}
result = await agent.ainvoke(messages, config=session_config)

In [8]:
result

{'messages': [HumanMessage(content='金沢に旅行に行きたい。さいたま市からの行き方を教えて。', additional_kwargs={}, response_metadata={}, id='c03e0532-60fa-4e80-aef6-2812aa63a215'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "\\u3055\\u3044\\u305f\\u307e\\u5e02\\u304b\\u3089\\u91d1\\u6ca2 \\u884c\\u304d\\u65b9"}'}, '__gemini_function_call_thought_signatures__': {'2943e147-cad7-4d93-84e6-24b17481a016': 'CocFAY89a1+E0uak3mKq3THEUwxfpTWMcBTPdu5kfleZ80BJkjBlfGU1q/H1QLr1c9z1so0GeSp1hqlOQ6J3fXtSqsQ+KLwnmYLXsk4W/D/SCkVLzmRruTM4ChVl3/o9Xc7EPXb0GLNFuCuME4P0Kn9q6RJHTJIW4Pz3n6XrkYdYZqPROmewo4lBi33dGqI3JaouMZn9Qe43YCsLlBqYBtI59ZSpSRMs6Wr2jO3cih4BWLhB/+z5Gk3VR2Yfd30eeXTrsKg3EkH4aGY+ClawyFUtztjsHb/yhJ+WHhSC6Qbmx9c4Psk4ZG/v9ApAqwPGgQuXeymBZ5MQdgjjyq9/4UTejdr58TVFtoU1fv+F1yZHrI2hldcE8ECT6/WkZe6g6j7GdXEPymbc8bfpT0C7FMfhr1VfpewMLVBZq/l8F2OR7EUT+V/fvq2K0IlQiYYhO7xaVzhwTj470c0Ib7jxZzX0T68FBGACfc4RBWAXBcoIkLivxQTe28gwLGuF86sou9wZj2JXPZ35tVa/W3GrG8keWen7uQiDl5Fz2pWirG3n9k

In [8]:
question = "美味しいものも教えて"
messages = {"messages": [{"role": "user", "content": question}]}
result = await agent.ainvoke(messages, config=session_config)

In [9]:
result

{'messages': [HumanMessage(content='金沢に旅行に行きたい。さいたま市からの行き方を教えて。', additional_kwargs={}, response_metadata={}, id='30a1bd61-f5b9-45e0-8b11-db2ac49c7970'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "\\u3055\\u3044\\u305f\\u307e\\u5e02\\u304b\\u3089\\u91d1\\u6ca2\\u3078\\u306e\\u884c\\u304d\\u65b9"}'}, '__gemini_function_call_thought_signatures__': {'b36a9b03-43bd-49cb-950f-04f4651554cf': 'CpcEAY89a1+Sa5ZN5SJjkXWdFlG86d0AU1ZOvo2IXjv/ymn44gSAFpSyHJYjZVNzHLy95w/n3PZMmcxNW7/0zxcmMylqyY4NvUe2hcDbQwkkT6lWrV/kSpBkLE+n3WNEe2d+1PnSJcPgOH3NSw0ss9PC6tZCpWo6mtbbi4BCsMxL4cIw5pp6xo+pv/iQsPSqmrMfK2dtIDVhu2GmV74pb50xm0XaiQ47NVs4RkWSFqLFD/2GiLiP17JpPrbARAROc1FGHtgOqKmwCEUYg8WlkLIG9P7sOtnK0BhV9+/mQ5VBSHCJrEjCsH5laG/Yc2IT0oao1m5zY0X1B3tIraAhj1hh475hHEfVi7IYL6L67bVlwEVjH1lgnD6Ath+lzpZ2IUgxTxfa4XW0/z1Vcwn49+2fysPaMkVOhi6KaZqmU3dQ6yCQer45+EFMyVODjUcQpmZwkUnVG1JssILc1P9EHsZmIBcpku0L7pcg+VD5hdUVg8ZOKml7C8JHVl/doWE8IydO+iqu9HszeATny0BabPrt3Z+USksD5